# A/A Test: Cross-Technique Comparison

Reads all A/A results from the shared BQ table and produces:
- Statistical validation (t-test + bootstrap CI) per technique/segment
- Pass/fail scorecard
- Stability analysis across time windows
- Visualizations for comparison

---
## 1. Config & Authentication

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_style('whitegrid')

from google.colab import auth
from google.cloud import bigquery

auth.authenticate_user()

# Mount Google Drive (one-time auth prompt)
from google.colab import drive
drive.mount('/content/drive')

# Point Python to the config directory on Drive
# Upload aa_config.py + campaign_data.csv to this folder ONCE
import sys, os
CONFIG_DIR = '/content/drive/MyDrive/aa_test'
sys.path.insert(0, CONFIG_DIR)
os.chdir(CONFIG_DIR)  # so aa_config can find campaign_data.csv

import aa_config as cfg

client = bigquery.Client(project=cfg.PROJECT_ID)
print(f'Connected to {cfg.PROJECT_ID}')

---
## 2. Load Results

In [ ]:
df = client.query(f"""
    SELECT * FROM `{cfg.AA_RESULTS_TABLE}`
    ORDER BY technique, country, window_start, seed, base_segment
""").to_dataframe()

print(f'Loaded {len(df)} rows from {cfg.AA_RESULTS_TABLE}')
print(f'Techniques: {df["technique"].unique().tolist()}')
print(f'Countries: {df["country"].unique().tolist()}')
print(f'Segments: {df["base_segment"].unique().tolist()}')
df.head()

---
## 3. Statistical Validation

In [ ]:
def stat_validation(values, label=''):
    values = np.array(values, dtype=float)
    values = values[~np.isnan(values)]
    if len(values) < 3:
        return {'label': label, 'mean': np.nan, 'std': np.nan, 'n': len(values),
                'p_value': np.nan, 'ci_lower': np.nan, 'ci_upper': np.nan,
                'ci_contains_zero': None}
    t_stat, p_value = stats.ttest_1samp(values, 0)
    rng = np.random.RandomState(42)
    boots = [rng.choice(values, size=len(values), replace=True).mean() for _ in range(10_000)]
    ci_lo, ci_hi = np.percentile(boots, [2.5, 97.5])
    return {'label': label, 'mean': values.mean(), 'std': values.std(),
            'n': len(values), 'p_value': p_value,
            'ci_lower': ci_lo, 'ci_upper': ci_hi,
            'ci_contains_zero': bool(ci_lo <= 0 <= ci_hi)}


def assess_verdict(abs_mean, ci_contains_zero):
    if np.isnan(abs_mean): return 'INSUFFICIENT_DATA'
    if abs_mean > cfg.BIAS_THRESHOLD_HARD: return 'HARD_FAIL'
    if abs_mean > cfg.BIAS_THRESHOLD_WARN: return 'WARNING'
    if ci_contains_zero is False: return 'WARNING'
    return 'PASS'

print('Validation functions loaded.')

---
## 4. Scorecard

In [ ]:
scorecard_rows = []
periods = ['pre_period_uplift', 'campaign_period_uplift', 'post_period_uplift']

for technique in sorted(df['technique'].unique()):
    for segment in sorted(df[df['technique']==technique]['base_segment'].unique()):
        sub = df[(df['technique']==technique) & (df['base_segment']==segment)]
        for period in periods:
            sv = stat_validation(sub[period].values, period)
            verdict = assess_verdict(
                abs(sv['mean']) if not np.isnan(sv['mean']) else np.nan,
                sv['ci_contains_zero'])
            scorecard_rows.append({
                'technique': technique, 'segment': segment, 'period': period,
                'n_runs': sv['n'], 'mean_uplift': sv['mean'],
                'p_value': sv['p_value'],
                'ci': f"[{sv['ci_lower']:.4f}, {sv['ci_upper']:.4f}]" if not np.isnan(sv['ci_lower']) else 'N/A',
                'ci_contains_zero': sv['ci_contains_zero'],
                'verdict': verdict})

df_scorecard = pd.DataFrame(scorecard_rows)

# Overall verdict
verdicts = df_scorecard['verdict'].values
if 'HARD_FAIL' in verdicts: overall = 'HARD_FAIL'
elif 'WARNING' in verdicts: overall = 'WARNING'
elif 'INSUFFICIENT_DATA' in verdicts: overall = 'INSUFFICIENT_DATA'
else: overall = 'PASS'

print(f'\nOVERALL VERDICT: {overall}\n')

# Show total-level scorecard
total_card = df_scorecard[df_scorecard['segment']=='total']
print(total_card[['technique','period','n_runs','mean_uplift','p_value','ci','verdict']].to_string(index=False))

---
## 5. Per-Segment Scorecard

In [ ]:
seg_card = df_scorecard[
    (df_scorecard['segment']!='total') & (df_scorecard['period']=='campaign_period_uplift')
]
if len(seg_card) > 0:
    pivot = seg_card.pivot_table(index='segment', columns='technique',
                                  values='verdict', aggfunc='first')
    print('Campaign-period verdict by segment:')
    print(pivot.to_string())
else:
    print('No segment-level data available.')

---
## 6. Stability Analysis

In [ ]:
for technique in sorted(df['technique'].unique()):
    sub = df[(df['technique']==technique) & (df['base_segment']=='total')]
    if len(sub) < 4: continue
    print(f'\n--- {technique} ---')
    ws = sub.groupby('window_start')['campaign_period_uplift'].agg(['mean','std','count'])
    print(ws.to_string())
    groups = [g['campaign_period_uplift'].dropna().values
              for _, g in sub.groupby('window_start')]
    groups = [g for g in groups if len(g) >= 2]
    if len(groups) >= 2:
        lev_stat, lev_p = stats.levene(*groups)
        print(f'Levene: stat={lev_stat:.4f}, p={lev_p:.4f} '
              f'{"WARNING" if lev_p<0.05 else "PASS"}')
        kw_stat, kw_p = stats.kruskal(*groups)
        print(f'Kruskal-Wallis: stat={kw_stat:.4f}, p={kw_p:.4f} '
              f'{"WARNING" if kw_p<0.05 else "PASS"}')

---
## 7. Visualizations

In [ ]:
# Uplift distributions per technique
df_total = df[df['base_segment']=='total']
techniques = sorted(df_total['technique'].unique())
n_tech = len(techniques)

if n_tech > 0:
    fig, axes = plt.subplots(1, max(n_tech, 1), figsize=(7*n_tech, 5), squeeze=False)
    for i, tech in enumerate(techniques):
        ax = axes[0][i]
        vals = df_total[df_total['technique']==tech]['campaign_period_uplift'].dropna()
        if len(vals) == 0:
            ax.set_title(f'{tech} (no data)')
            continue
        sns.histplot(vals, kde=True, ax=ax, bins=15, color='steelblue')
        ax.axvline(x=0, color='red', ls='--', lw=2, label='Expected (0)')
        ax.axvline(x=vals.mean(), color='orange', ls='-', lw=2, label=f'Mean: {vals.mean():.4f}')
        ax.set_title(f'{tech}')
        ax.set_xlabel('Campaign Period Uplift')
        ax.legend(fontsize=9)
    plt.suptitle('A/A Test: Campaign-Period Uplift by Technique', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
# Cross-technique box plot
if n_tech > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    df_total.boxplot(column='campaign_period_uplift', by='technique', ax=ax)
    ax.axhline(y=0, color='red', ls='--', lw=1.5)
    ax.axhline(y=cfg.BIAS_THRESHOLD_WARN, color='orange', ls=':', alpha=0.7)
    ax.axhline(y=-cfg.BIAS_THRESHOLD_WARN, color='orange', ls=':', alpha=0.7)
    ax.set_title('Campaign-Period Uplift: Cross-Technique Comparison')
    ax.set_ylabel('Uplift')
    ax.set_xlabel('Technique')
    plt.suptitle('')
    plt.tight_layout()
    plt.show()

In [ ]:
# Segment heatmap per technique
for tech in techniques:
    sub = df[(df['technique']==tech) & (df['base_segment']!='total')]
    if len(sub) == 0: continue
    heat = sub.groupby('base_segment')[['pre_period_uplift','campaign_period_uplift','post_period_uplift']].mean()
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(heat, annot=True, fmt='.4f', cmap='RdYlGn_r', center=0,
                ax=ax, linewidths=0.5,
                xticklabels=['Pre-Period','Campaign','Post-Period'])
    ax.set_title(f'{tech}: Mean Uplift by Segment (0 = no bias)')
    plt.tight_layout()
    plt.show()

---
## 8. Summary Report

In [ ]:
print('='*60)
print('A/A TEST COMPARISON REPORT')
print('='*60)
print(f'\nTechniques: {techniques}')
print(f'Countries: {df["country"].unique().tolist()}')
for tech in techniques:
    sub = df_total[df_total['technique']==tech]
    print(f'\n--- {tech} ({len(sub)} runs) ---')
    for col, label in [('pre_period_uplift','Pre'),('campaign_period_uplift','Campaign'),('post_period_uplift','Post')]:
        vals = sub[col].dropna()
        if len(vals)>0:
            print(f'  {label}: mean={vals.mean():+.4f} std={vals.std():.4f} [{vals.min():.4f}, {vals.max():.4f}]')
    cov = sub['exact_match_pct'].dropna()
    if len(cov)>0:
        print(f'  Exact match: {cov.mean():.1%} (std: {cov.std():.1%})')
print(f'\n{"="*60}')
print(f'OVERALL VERDICT: {overall}')
print(f'{"="*60}')